In [16]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # repo root, since notebook lives in notebooks/

from src.evaluation.viz import floater_cloud


In [19]:
import os, glob, re
import numpy as np
import pandas as pd
import open3d as o3d
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# --- config ---
RESULTS_DIR = '/Users/adam/Documents/MILA/projects/depth-uncertainty-slam/results/office0_100'   # dir containing {model}_{init,final}_cloud.npz
MESH_PATH   = "/Users/adam/Documents/MILA/projects/depth-uncertainty-slam/results/office0_mesh.ply" # e.g. '/home/adam/scratch/datasets/replica/Replica/office0/mesh.ply'; None to skip overlay
TAU         = 0.05                    # floater threshold in meters

COLORS_RGB = {'surface': 'rgb(150,150,150)', 'clean': 'rgb(255,140,0)', 'amb': 'rgb(230,15,15)'}
COLORS_O3D = {'surface': [0.55,0.55,0.55], 'clean': [1.0,0.55,0.0], 'amb': [0.9,0.06,0.06]}

In [20]:
def load_artifacts(results_dir):
    arts = {}
    for f in sorted(glob.glob(os.path.join(results_dir, '*_cloud.npz'))):
        name = os.path.basename(f)[:-len('_cloud.npz')]
        m = re.match(r'(.+)_(init|final)$', name)
        if not m:
            continue
        model, stage = m.group(1), m.group(2)
        z = np.load(f)
        arts[(model, stage)] = dict(points=z['points'], dist=z['dist'],
                                    amb=z['amb'].astype(bool), tau=float(z['tau']))
    return arts

arts = load_artifacts(RESULTS_DIR)
print('loaded:', sorted(arts.keys()))

loaded: [('bimodal', 'final'), ('bimodal', 'init')]


In [21]:
def stats_row(d, tau):
    return dict(n=len(d), median=np.median(d), p95=np.percentile(d, 95),
                p99=np.percentile(d, 99), max=d.max(),
                fr_5cm=(d > 0.05).mean(), fr_10cm=(d > 0.10).mean())

rows = []
for (model, stage), A in sorted(arts.items()):
    for sub, m in [('all', np.ones(len(A['dist']), bool)), ('amb', A['amb']), ('clean', ~A['amb'])]:
        if m.any():
            rows.append(dict(model=model, stage=stage, subset=sub, **stats_row(A['dist'][m], TAU)))

df = pd.DataFrame(rows)
df.style.format({'median':'{:.4f}','p95':'{:.4f}','p99':'{:.4f}','max':'{:.4f}',
                 'fr_5cm':'{:.3f}','fr_10cm':'{:.3f}'})

,model,stage,subset,n,median,p95,p99,max,fr_5cm,fr_10cm
0,bimodal,final,all,3088,0.0027,0.0506,0.1957,0.7954,0.051,0.021
1,bimodal,final,amb,130,0.0169,0.2465,0.3905,0.4316,0.277,0.100
2,bimodal,final,clean,2958,0.0026,0.0394,0.1524,0.7954,0.041,0.017
3,bimodal,init,all,3088,0.0073,0.1217,0.3364,2.5439,0.134,0.063
4,bimodal,init,amb,130,0.0413,0.2760,0.4285,0.4785,0.454,0.215
5,bimodal,init,clean,2958,0.0070,0.1098,0.3098,2.5439,0.120,0.057


In [22]:
def _add_traces(fig, A, tau, row, col, showlegend, size):
    pts, amb = A['points'], A['amb']
    fl = A['dist'] > tau
    groups = [('on-surface', ~fl, COLORS_RGB['surface']),
              ('floater (clean)', fl & ~amb, COLORS_RGB['clean']),
              ('floater (ambiguous)', fl & amb, COLORS_RGB['amb'])]
    for name, mask, color in groups:
        p = pts[mask]
        fig.add_trace(go.Scatter3d(x=p[:,0], y=p[:,1], z=p[:,2], mode='markers',
            marker=dict(size=size, color=color), name=name, legendgroup=name,
            showlegend=showlegend), row=row, col=col)

def _panel_grid(keys, titles, tau, size):
    n = len(keys)
    fig = make_subplots(rows=1, cols=n, specs=[[{'type':'scene'}]*n], subplot_titles=titles)
    for i, k in enumerate(keys):
        _add_traces(fig, arts[k], tau, 1, i+1, showlegend=(i==0), size=size)
    for s in [f'scene{i+1 if i else ""}' for i in range(n)]:
        fig.layout[s].aspectmode = 'data'
    fig.update_layout(height=560, margin=dict(l=0,r=0,t=30,b=0),
                      legend=dict(itemsizing='constant'))
    return fig

def compare_stages(model, tau=TAU, size=1.6):
    keys = [(model,'init'), (model,'final')]
    keys = [k for k in keys if k in arts]
    return _panel_grid(keys, [f'{m} / {s}' for m,s in keys], tau, size)

def compare_models(stage='final', tau=TAU, size=1.6):
    models = sorted({m for (m,s) in arts if s==stage})
    keys = [(m, stage) for m in models]
    return _panel_grid(keys, [f'{m} / {stage}' for m,_ in keys], tau, size)

compare_stages('bimodal').show()

In [24]:
def o3d_cloud(A, tau):
    pc = o3d.geometry.PointCloud()
    pc.points = o3d.utility.Vector3dVector(A['points'].astype(float))
    c = np.tile(COLORS_O3D['surface'], (len(A['points']), 1))
    fl = A['dist'] > tau
    c[fl & ~A['amb']] = COLORS_O3D['clean']
    c[fl &  A['amb']] = COLORS_O3D['amb']
    pc.colors = o3d.utility.Vector3dVector(c)
    return pc

def view(model='bimodal', stage='final', tau=TAU, mesh_path=MESH_PATH):
    pc = o3d_cloud(arts[(model, stage)], tau)
    geoms = [pc]
    if mesh_path:
        mesh = o3d.io.read_triangle_mesh(mesh_path)
        if len(mesh.triangles) > 200_000:
            mesh = mesh.simplify_quadric_decimation(150_000)
        mesh.compute_vertex_normals()
        mesh.paint_uniform_color([0.85, 0.85, 0.85])
        geoms = [mesh, pc]
    o3d.visualization.draw_geometries(geoms, window_name=f'{model}/{stage}  (red=amb floater)')

view('bimodal', 'final')

[Open3D WARNING] Read PLY failed: A polygon in the mesh could not be decomposed into triangles.
[Open3D WARNING] Read PLY failed: unable to read file: /Users/adam/Documents/MILA/projects/depth-uncertainty-slam/results/office0_mesh.ply


RPly: Aborted by user
